In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from models.detection_cnn import DASCountCNN
from models.detection_transformer import DASCountTransformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [22]:
# ---------------------------------------------------------
# Cell 2: Data Loading and Path Setup
# ---------------------------------------------------------
# Load the CSV file
# Note: If your CSV is tab-separated (like the text you pasted earlier), use sep='\t'
df = pd.read_csv('data/corrected_labels.csv')

# Fix paths: ensure they point to the denoised files
# This formats sample_id (e.g., 0) to "data/denoised/denoised_sample_000000.npy"
df['data_path'] = df['sample_id'].apply(
    lambda x: f"data/denoised/denoised_sample_{str(x).zfill(6)}.npy"
)

print(f"Total samples available: {len(df)}")
display(df.head())

Total samples available: 113


,sample_id,data_path,count,start_frame,end_frame,vehicle_type
0,0,data/denoised/denoised_sample_000000.npy,4,1873,2023,mixed
1,1,data/denoised/denoised_sample_000001.npy,4,1902,2053,mixed
2,2,data/denoised/denoised_sample_000002.npy,3,1932,2083,mixed
3,3,data/denoised/denoised_sample_000003.npy,2,1962,2113,mixed
4,4,data/denoised/denoised_sample_000004.npy,1,1992,2143,suv


In [23]:
# --- Replace Cell 3 ---
# ---------------------------------------------------------
# Cell 3: PyTorch Dataset & Split
# ---------------------------------------------------------
class DASCountDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)
        # Map string classes to integers
        self.type_map = {'background': 0, 'mixed': 1, 'suv': 2, 'van': 3, 'sedan': 4, 'truck': 5}
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load denoised numpy array
        x_np = np.load(row['data_path']).astype(np.float32)
        x_tensor = torch.from_numpy(x_np).unsqueeze(0)
        
        # 1. Target for count (float for MSE)
        y_count_tensor = torch.tensor([row['count']], dtype=torch.float32)
        
        # 2. Target for vehicle type (long integer for CrossEntropy)
        type_str = row['vehicle_type'].lower()
        y_type_tensor = torch.tensor(self.type_map[type_str], dtype=torch.long)
        
        return x_tensor, y_count_tensor, y_type_tensor

dataset = DASCountDataset(df)

# Train/Val/Test Split (70% Train / 15% Val / 15% Test)
total_size = len(dataset)
train_size = int(0.7 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

train_ds, val_ds, test_ds = random_split(
    dataset, [train_size, val_size, test_size], 
    generator=torch.Generator().manual_seed(42) # Seed for reproducibility
)

batch_size = 8
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

print(f"Split sizes -> Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

Split sizes -> Train: 79 | Val: 16 | Test: 18


In [24]:
# --- Replace Cell 4 ---
# Grab one sample from your dataset to check its shape
sample_x, _, _ = train_ds[0] # Note: we unpack 3 items now
num_spatial_channels = sample_x.shape[1] 

print(f"Detected {num_spatial_channels} spatial channels.")

# Initialize the Transformer
model = DASCountTransformer(
    spatial_channels=num_spatial_channels, 
    d_model=64,   
    nhead=4,      
    num_layers=2  
).to(device)

# Initialize two separate loss functions
count_criterion = nn.MSELoss()             # For predicting the number
type_criterion = nn.CrossEntropyLoss()     # For predicting the category
optimizer = optim.Adam(model.parameters(), lr=1e-4)

Detected 34 spatial channels.


In [ ]:
# Load the trained model saved by: python train.py --task detection
model.load_state_dict(torch.load("results/detection/best_model.pt", map_location=device))
model.eval()
print("Model loaded from results/detection/best_model.pt")

In [26]:
# --- Replace Cell 6 ---
# ---------------------------------------------------------
# Cell 6: Evaluation with Advanced Counting Metrics & Logic
# ---------------------------------------------------------
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score

# Reverse mapping to turn integers back to strings
IDX_TO_TYPE = {0: 'background', 1: 'mixed', 2: 'suv', 3: 'van', 4: 'sedan', 5: 'truck'}

def enforce_business_logic(raw_count_pred, type_logits):
    """Enforces rules: count=0 -> background, count>1 -> mixed, count=1 -> (suv|van|sedan|truck)"""
    final_count = int(round(raw_count_pred))
    
    if final_count == 0:
        final_type = 'background'
    elif final_count > 1:
        final_type = 'mixed'
    else: # final_count == 1
        # Extract logits only for indices 2, 3, 4, 5
        single_vehicle_logits = type_logits[2:6]
        # Find local max index (0 to 3) and add 2 to match original indices
        best_single_idx = np.argmax(single_vehicle_logits) + 2
        final_type = IDX_TO_TYPE[best_single_idx]
        
    return final_count, final_type

model.eval()
actual_counts, pred_counts = [], []
actual_types, pred_types = [], []

with torch.no_grad():
    for x, y_count, y_type in test_loader:
        x = x.to(device)
        p_counts, p_types = model(x)
        
        # Move to CPU for processing
        p_counts = p_counts.cpu().numpy().flatten()
        p_types = p_types.cpu().numpy()
        y_count = y_count.numpy().flatten()
        y_type = y_type.numpy().flatten()
        
        for i in range(len(x)):
            actual_counts.append(y_count[i])
            actual_types.append(IDX_TO_TYPE[y_type[i]])
            
            # Apply our strict rules!
            final_c, final_t = enforce_business_logic(p_counts[i], p_types[i])
            pred_counts.append(final_c)
            pred_types.append(final_t)

print("--- Test Set Predictions ---")
for act_c, act_t, pred_c, pred_t in zip(actual_counts, actual_types, pred_counts, pred_types):
    print(f"Actual: {int(act_c)} ({act_t:10s}) | Predicted: {pred_c} ({pred_t})")

# --- Calculate Metrics ---
actuals_arr = np.array(actual_counts)
preds_arr = np.array(pred_counts)

exact_acc = np.mean(actuals_arr == preds_arr) * 100
off_by_one_acc = np.mean(np.abs(actuals_arr - preds_arr) <= 1) * 100
mae = mean_absolute_error(actuals_arr, preds_arr)
rmse = np.sqrt(mean_squared_error(actuals_arr, preds_arr))
bias = np.mean(preds_arr - actuals_arr)

# Type classification accuracy
type_acc = accuracy_score(actual_types, pred_types) * 100

print("\n--- Final Metrics ---")
print(f"Vehicle Type Accuracy: {type_acc:.1f}%")
print(f"Count Exact Match:     {exact_acc:.1f}%")
print(f"Count Off-by-One:      {off_by_one_acc:.1f}%")
print(f"Count MAE:             {mae:.2f} events")
print(f"Count RMSE:            {rmse:.2f} events")
print(f"Count Bias:            {bias:.2f}")

--- Test Set Predictions ---
Actual: 1 (suv       ) | Predicted: 1 (suv)
Actual: 1 (suv       ) | Predicted: 1 (suv)
Actual: 0 (background) | Predicted: 0 (background)
Actual: 1 (sedan     ) | Predicted: 1 (suv)
Actual: 0 (background) | Predicted: 0 (background)
Actual: 0 (background) | Predicted: 0 (background)
Actual: 4 (mixed     ) | Predicted: 3 (mixed)
Actual: 0 (background) | Predicted: 0 (background)
Actual: 4 (mixed     ) | Predicted: 3 (mixed)
Actual: 2 (mixed     ) | Predicted: 3 (mixed)
Actual: 0 (background) | Predicted: 0 (background)
Actual: 3 (mixed     ) | Predicted: 3 (mixed)
Actual: 0 (background) | Predicted: 0 (background)
Actual: 1 (suv       ) | Predicted: 2 (mixed)
Actual: 3 (mixed     ) | Predicted: 3 (mixed)
Actual: 0 (background) | Predicted: 0 (background)
Actual: 3 (mixed     ) | Predicted: 3 (mixed)
Actual: 2 (mixed     ) | Predicted: 1 (truck)

--- Final Metrics ---
Vehicle Type Accuracy: 83.3%
Count Exact Match:     72.2%
Count Off-by-One:      100.0%
Cou

In [ ]:
import matplotlib.pyplot as plt
from Utilities import plot_das_data
from config import DAS_FILE
from DAS import DAS

das = DAS(DAS_FILE)
dx = das.meta['dx']
dt = das.meta['dt']

sample_idx = 16
x_tensor, y_count_tensor, y_type_tensor = test_ds[sample_idx]

model.eval()
with torch.no_grad():
    x_input = x_tensor.unsqueeze(0).to(device)
    raw_count_tensor, type_logits_tensor = model(x_input)

raw_count = raw_count_tensor.item()
type_logits = type_logits_tensor.cpu().numpy().flatten()
pred_count, pred_type = enforce_business_logic(raw_count, type_logits)

actual_count = int(y_count_tensor.item())
actual_type_idx = int(y_type_tensor.item())
actual_type = {v: k for k, v in dataset.type_map.items()}[actual_type_idx]

plot_data = x_tensor.squeeze().numpy()
channels = np.arange(plot_data.shape[0])

fig, ax = plt.subplots(figsize=(12, 6))
plot_title = f"Actual: {actual_count} ({actual_type}) | Predicted: {pred_count} ({pred_type}) | Raw Count: {raw_count:.2f}"
plot_das_data(data=plot_data, channels=channels, dx=dx, dt=dt,
              title=plot_title, ax=ax, fig=fig, show=False)
plt.tight_layout()
plt.show()